# Informace
Pro školky, kdy jsme šli ještě o level níž

In [1]:
import pandas as pd

In [2]:
original = pd.read_csv('original_skolky_with_fee.csv')

In [4]:
part1 = pd.read_excel("03_validated_skolky/skolky_with_fee_v1_validated.xlsx", index_col=0)
part2 = pd.read_excel("03_validated_skolky/skolky_with_fee_v2_validated.xlsx", index_col=0)

Sjednocení sloupců

In [5]:
part1 = part1.rename(columns={'fee_validated': 'monthly_fee'})
part2 = part2.rename(columns={'fee_validated': 'monthly_fee'})

In [6]:
# select column monthly fee, if there is a value invalid, replace it wirth nan
part1["monthly_fee"] = part1["monthly_fee"].replace("invalid", None).astype('float')
part2["monthly_fee"] = part2["monthly_fee"].replace("invalid", None).astype('float')


Sloučení všech tabulek

In [7]:
all_fee = pd.concat([part1, part2])


In [9]:
found_fee = all_fee[all_fee['monthly_fee'].notnull()]
missing_fee = all_fee[~all_fee['monthly_fee'].isin(found_fee['monthly_fee'])]

In [44]:
found_fee.to_csv('found_fee_v2.csv', index=False)
missing_fee.to_csv('missing_fee_v2.csv', index=False)

In [ ]:
merge_fee = all_fee.drop(columns=['WWW','fee', 'sentence', 'found_by_scraping'])

In [25]:
merge_fee['sentence'].notnull().sum()

np.int64(695)

In [28]:
original = original[['RED_IZO', 'IČO', 'Zřizovatel', 'Území', 'Kraj', 'Okres/Obvod',
       'Správní úřad', 'ORP', 'Název ORP', 'Plný název', 'Zkrácený název',
       'Ulice', 'Č. p.', 'Č. or.', 'Část obce', 'PSČ', 'Místo','WWW', 'ZUJ', 'monthly_fee', 'meals']]


In [29]:
original['monthly_fee'].notnull().sum()

np.int64(1144)

In [30]:
original_with_fee = pd.merge(original, merge_fee, left_on='Plný název', right_on='Plný název', how='left')
original_with_fee

,RED_IZO,IČO,Zřizovatel,Území,Kraj,Okres/Obvod,Správní úřad,ORP,Název ORP,Plný název,...,Č. or.,Část obce,PSČ,Místo,WWW,ZUJ,monthly_fee_x,meals,sentence,monthly_fee_y
0,600000206,49625918,6,CZ0101,Hlavní město Praha,Praha 1,B11000,1100,Hlavní město Praha,Mateřská škola sv. Voršily v Praze,...,11.0,Nové Město,11000,Praha,https://mssvvorsily.cz/,554782.0,2500.0,NaN,NaN,NaN
1,600000222,61379310,6,CZ0104,Hlavní město Praha,Praha 4,B11000,1100,Hlavní město Praha,Církevní mateřská škola Studánka,...,2.0,Kamýk,14200,Praha,http://www.cms-studanka.cz,554782.0,1700.0,NaN,NaN,NaN
2,600000231,25765710,5,CZ0104,Hlavní město Praha,Praha 4,B11000,1100,Hlavní město Praha,Modrý klíč - základní škola speciální a mateřs...,...,2.0,Kamýk,14200,Praha,https://www.webskoly.cz/modryklic,554782.0,NaN,NaN,NaN,NaN
3,600000249,60437171,6,CZ0105,Hlavní město Praha,Praha 5,B11000,1100,Hlavní město Praha,Církevní mateřská škola Srdíčko,...,2.0,Stodůlky,15500,Praha,www.cms-srdicko.cz,554782.0,1300.0,NaN,NaN,NaN
4,600000257,25143701,5,CZ0105,Hlavní město Praha,Praha 5,B11000,1100,Hlavní město Praha,Bilingvální mateřská škola pro sluchově postiž...,...,22.0,Stodůlky,15500,Praha,http://www.pipan.cz,554782.0,4000.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5459,691018308,5032695,5,CZ0321,Plzeňský kraj,Domažlice,B32000,3202,Domažlice,Mateřská škola Loďka s.r.o.,...,NaN,Bezděkovské Předměstí,34401,Domažlice,NaN,553425.0,NaN,NaN,NaN,NaN
5460,691018341,21162166,5,CZ0712,Olomoucký kraj,Olomouc,B71000,7107,Olomouc,Maple Bear mateřská škola Olomouc s.r.o.,...,1.0,Hodolany,77900,Olomouc,https://www.olomouc.maplebear.cz/,500496.0,NaN,NaN,"[('3', ""Souhlasíte s používáním cookies?\\n\\u...",15000.0
5461,691018367,21621357,2,CZ0423,Ústecký kraj,Litoměřice,D42050,4205,Litoměřice,Lesní mateřská škola Hliněnka,...,7.0,Litoměřice-Město,41201,Litoměřice,NaN,564567.0,NaN,NaN,NaN,NaN
5462,691018391,21757194,5,CZ0102,Hlavní město Praha,Praha 2,B11000,1100,Hlavní město Praha,"Mateřská škola Eda, s.r.o.",...,40.0,Vinohrady,12000,Praha,NaN,554782.0,NaN,NaN,NaN,NaN


In [31]:
# if monthly fee y is not nan, then insert  the value into column monthly_fee_x
original_with_fee['monthly_fee_x'] = original_with_fee.apply(
    lambda row: row['monthly_fee_y'] if pd.notnull(row['monthly_fee_y']) else row['monthly_fee_x'], axis=1)
# drop the monthly_fee_y column
original_with_fee = original_with_fee.drop(columns=['monthly_fee_y'])
# rename the monthly_fee_x column to monthly_fee
original_with_fee = original_with_fee.rename(columns={'monthly_fee_x': 'monthly_fee'})

In [32]:
obce = original_with_fee[original_with_fee["Zřizovatel"] == 2]

In [34]:
obce['sentence'].notnull().sum()

np.int64(585)

In [ ]:
original_with_fee.to_csv('original_skolky_with_fee_v2.csv', index=False)